# READ FIRST: this notebook was re-purposed for the wrapper kill test (Nov 11, 2026)

The sealed recipe below (environment, load ladder, sanity check, feed
builder) is unchanged from the MiniCPM battery notebook. What changed
is only the job list: instead of the old battery, the runner asks the
wrapper kill test. Design frozen in
`kill-test-wrapper/DESIGN-FROZEN-MINICPM.md` in the Project K files.

**Clips (4):** `swap_water.mp4`, `swap_vacuum.mp4`,
`docs_applause_still.mp4`, `swap_silence.mp4` (the control). The swap
files are in the Drive `project-k-clips` folder; the applause file may
need a hand upload into the session folder `/content/project-k-clips`.

**Arms per sound clip:** S (standard), R1 to R3 (ordinary re-asks),
W1 (sound-only), W2 (yes/no own sound), LURE (yes/no wrong sound),
then W3, the carry-forward final built from that clip's own W1 answer.
Silence gets S, W1, W2, W3. 28 asks, 1 repeat each, greedy.

**Output:** `killtest_minicpm_results.csv`, saved to the Drive
`project-k-clips` folder and to `/content`. Any older battery-era text
further down in this notebook (clip lists, Part A / Part B talk) is
leftover wording from the original notebook and does NOT describe
this run. The runner's own prints and the CSV are the record.

# Project K, fourth model: MiniCPM-o 2.6 hears the same clips

**Why this notebook exists.** Three models have already faced these clips
with the same questions. Pegasus 1.6, through the Twelve Labs Analyze
door, reported no non-speech sound anywhere, and its answers did not
change when the sound was muted. Qwen2.5-Omni-3B heard the jar's running
water and a beep, and missed the eggs. Qwen2.5-Omni-7B heard more real
sound than its smaller sibling, the eggs included, but guessed kitchen
noise on the vacuum clip and failed the plain question's easiest control,
which voided its bed test.

**What this notebook does.** It runs a fourth model from a different
family, MiniCPM-o 2.6 (8B, built by OpenBMB), on the exact same files
with the exact same questions, so the four can be compared cell by cell.

**Where it runs.** This notebook is built for the paid Colab account and
an **L4 GPU**. The model in full 16-bit form wants about 17 GB of chip
memory. A free T4 has 15.6 GB in total, so full form does not fit there.
An L4 has 24 GB, and full form fits with room to spare.

**How it runs, and why it changed.** Twice, this notebook tried to
install the model's pinned software stack into Colab's shared Python,
and twice the pinned versions never landed where the session reads:
Colab's own preloaded stack fought the install and won. The pins
themselves were then proven innocent on a clean Python 3.12 machine,
where the exact list installs without a fight. So this notebook no
longer touches Colab's Python. Step 1 builds a private virtual
environment, installs the model's stack inside it, and proves the
right versions are live there. Every later model step runs as a plain
script under that private Python, with its output streamed into this
notebook. Colab's environment cannot shadow, block, or confuse any of
it, because none of it is Colab's environment anymore.

**The one rule.** We claim only what we ran. Every result here is about
these clips, this model, this run. Nothing wider.

**How to run it.** Run the cells from the top, one by one, in order.
Step 1 builds the private environment and proves the stack inside it.
Step 2 gets the clips from your Drive. Step 3 writes the runner script,
then runs its check mode: the loading ladder and the sanity question.
Step 4 runs the battery and saves the answers as
`killtest_minicpm_results.csv` in the same Drive folder the clips came from.
No restart is needed anywhere in this notebook. The notebook's own
session never imports the model's stack, so there is nothing stale for
a restart to clear.

## Step 1 · Setup: a private environment for the model's stack

The next cell does four things:

1. Installs `uv`, a fast environment and package tool, if this machine
   does not already have it. That is the only thing it adds to Colab's
   own Python, and it is a tool, not part of the model's stack.
2. Creates a private virtual environment at `/content/minicpm-venv`,
   on Python 3.12, the same Python generation Colab itself runs. The
   model's stack was proven to install cleanly on Python 3.12, so the
   private environment keeps 3.12 and changes only who owns it.
3. Installs the model's stack inside that environment, in two passes.
   First, torch and torchaudio pinned to 2.8.0 from PyTorch's CUDA 12.6
   shelf: that is the exact build this runtime already runs, so the
   chip driver match is the proven one, and the installer cannot drag
   in a multi-gigabyte CUDA 13 torch the driver may not support (left
   floating, the resolver does exactly that). Second, the model card's
   own pin list, unchanged: transformers 4.44.2, accelerate 0.34.2,
   bitsandbytes 0.46.1, librosa 0.10.2.post1 (a newer patch than the
   card lists, because the card's version predates Colab's current
   numpy), soundfile, vocos, and vector-quantize-pytorch. Pillow rides
   along because the model's code and the battery both need it and
   nothing in the pin list pulls it in. vocos and
   vector-quantize-pytorch are parts the model's code imports for its
   speech side; they are installed even though this test never asks
   the model to speak. One more install rides after those two passes:
   flash-attn 2.8.3, the attention library the model's own modeling
   file imports. Transformers checks that file's imports before it
   loads a single weight, so without flash-attn the model cannot
   even start. Colab's stock environment carries it already, which
   is why the model card never lists it; the private environment
   gets its own copy as the official prebuilt wheel for this exact
   stack (Python 3.12, torch 2.8, CUDA 12, cxx11 ABI), pinned by its
   full download address, never a build from source.
4. Proves the stack, in a fresh process under the environment's own
   Python. The proof prints the Python version, the torch version, the
   transformers version and the file it loads from, and it imports
   `WHISPER_ATTENTION_CLASSES` from the transformers whisper module,
   the exact symbol the model's own code asks for and newer
   transformers versions removed, and it imports flash_attn and
   prints its version. A version number alone is not proof anymore;
   the imports are. If any line of the proof is wrong, Step 1 stops
   on the spot and the run goes no further.

Nothing in this cell installs over, downgrades, or removes anything
Colab shipped. The two failed runs of this notebook both died in that
fight; this design simply refuses to have it.

The install downloads a few gigabytes (torch is the big one) and takes
several minutes. That is normal. Every command is printed before it
runs, and the installers stream their output into the cell, so nothing
about this step happens off the record.

In [ ]:
# --- Setup: build the private environment and prove the stack inside it --------
# This cell never installs the model's stack into Colab's Python. Two
# earlier runs of this notebook died doing exactly that: the pinned
# transformers never landed where the session reads. The stack now
# lives in a private virtual environment at /content/minicpm-venv, and
# Step 1 ends by proving, in a fresh process under that environment's
# Python, that the pinned versions and the one symbol the model's code
# needs are really there.
import os, sys, shutil, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

VENV = "/content/minicpm-venv"
VENV_PY = os.path.join(VENV, "bin", "python")

def run_stream(cmd):
    print("+", " ".join(cmd))
    proc = subprocess.run(cmd)
    print("  exit code:", proc.returncode)
    return proc.returncode

# 1. uv: the environment tool. The only addition to Colab's own Python.
if shutil.which("uv") is None:
    print("uv not found. Installing it (a tool only, not the model stack).")
    rc = run_stream([sys.executable, "-m", "pip", "install", "uv"])
    if rc != 0 or shutil.which("uv") is None:
        print()
        print("!" * 72)
        print("STOP. uv could not be installed, so the private environment")
        print("cannot be built. Show this output to Muse.")
        print("!" * 72)
        raise SystemExit("STOP: uv install failed")
print("uv:", shutil.which("uv"))

# 2. The private environment itself, on Python 3.12. The card's stack
# was proven to install cleanly on Python 3.12 (a full clean resolution
# of the exact pin list), so the private environment keeps 3.12 and
# changes only who owns the environment: this run, not Colab.
if not os.path.exists(VENV_PY):
    rc = run_stream(["uv", "venv", "--python", "3.12", VENV])
    if rc != 0:
        print()
        print("!" * 72)
        print("STOP. The private environment could not be created.")
        print("Show this output to Muse.")
        print("!" * 72)
        raise SystemExit("STOP: venv creation failed")
ver = subprocess.run([VENV_PY, "--version"], capture_output=True, text=True)
print("venv python:", (ver.stdout or ver.stderr).strip())

# 3a. torch first, pinned to the runtime's own proven build. Colab's
# pip list on the failed runs showed torch 2.8.0+cu126: that is the
# build this chip and driver already run. Left floating, the resolver
# picks a newest torch built on CUDA 13 and downloads gigabytes of
# driver libraries the runtime may not support. Pinning it is not a
# change to the model card's list; the card leaves torch to the
# machine, and this makes the machine's proven choice explicit.
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 "torch==2.8.0", "torchaudio==2.8.0",
                 "--index-url", "https://download.pytorch.org/whl/cu126"])
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The torch install inside the private environment failed.")
    print("The installer's own output is above. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: torch install failed in venv")

# 3b. The model card's pin list, unchanged, in one resolver pass. No
# force-reinstall: the environment is empty and private, so there is
# nothing to force against. Pillow is added because the model's code
# and the battery import it and no package in the list pulls it in.
CARD_PACKAGES = [
    "transformers==4.44.2",
    "accelerate==0.34.2",
    "bitsandbytes==0.46.1",
    "librosa==0.10.2.post1",
    "soundfile==0.13.1",
    "vocos==0.1.0",
    "vector-quantize-pytorch==1.18.5",
    "Pillow",
]
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY] + CARD_PACKAGES)
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The model card's stack did not install inside the private")
    print("environment. The installer's own output is above, complete.")
    print("Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: card stack install failed in venv")

# 3c. flash-attn: the one package the sealed environment was missing.
# MiniCPM's own modeling file imports flash_attn, and transformers
# checks that file's imports before it loads a single weight. The
# fifth round's full-capture log (minicpm_load_log.txt) shows all
# three rungs dying in about three seconds on exactly that check:
# "This modeling file requires the following packages that were not
# found in your environment: flash_attn". Colab's stock environment
# carries flash-attn 2.8.3 already, which is why the model card never
# lists it; the private environment has to be given its own copy. It
# goes in as the official prebuilt wheel, never a build from source:
# Dao-AILab's flash-attn 2.8.3 for CPython 3.12, torch 2.8, and CUDA
# 12, in the cxx11 ABI TRUE form, which is the ABI torch 2.8 itself
# ships with (PyTorch dropped its pre-cxx11 ABI builds at 2.8). The
# address was fetched and the wheel opened when this round was
# built: it answers HTTP 200, it holds the cp312 CUDA extension, and
# its version reads 2.8.3. Its one extra dependency, einops, is
# small and comes from the normal shelf in the same pass.
FLASH_ATTN_WHEEL = ("https://github.com/Dao-AILab/flash-attention/"
                    "releases/download/v2.8.3/flash_attn-2.8.3"
                    "%2Bcu12torch2.8cxx11abiTRUE-cp312-cp312"
                    "-linux_x86_64.whl")
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 FLASH_ATTN_WHEEL])
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The flash-attn wheel did not install inside the private")
    print("environment. The installer's own output is above, complete.")
    print("Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: flash-attn install failed in venv")

# 4. Proof, in a fresh process under the environment's Python. Version
# alone is not accepted as proof anymore: the proof also imports
# WHISPER_ATTENTION_CLASSES, the exact symbol MiniCPM's own code asks
# transformers for, whose absence is what a wrong transformers version
# actually breaks, and flash_attn, the package the model's modeling
# file demands at the door (see 3c above).
PROOF = (
    "import sys, torch, transformers\n"
    "print('venv python:', sys.version.split()[0])\n"
    "print('torch version:', torch.__version__)\n"
    "print('transformers version:', transformers.__version__)\n"
    "print('transformers file:', transformers.__file__)\n"
    "from transformers.models.whisper.modeling_whisper import "
    "WHISPER_ATTENTION_CLASSES\n"
    "print('WHISPER_ATTENTION_CLASSES import: OK', "
    "sorted(WHISPER_ATTENTION_CLASSES.keys()))\n"
    "import flash_attn\n"
    "print('flash_attn version:', flash_attn.__version__)\n"
)
proof = subprocess.run([VENV_PY, "-c", PROOF],
                       capture_output=True, text=True)
print()
print(proof.stdout)
if proof.stderr.strip():
    print("proof process also said:")
    print("\n".join(proof.stderr.splitlines()[-10:]))
ok = ("transformers version: 4.44.2" in proof.stdout
      and "WHISPER_ATTENTION_CLASSES import: OK" in proof.stdout
      and "flash_attn version: 2.8.3" in proof.stdout)
if not ok:
    print("!" * 72)
    print("STOP. THE PROOF FAILED INSIDE THE PRIVATE ENVIRONMENT.")
    print("Either a version is wrong (transformers 4.44.2, flash_attn")
    print("2.8.3) or one of the proof imports failed: the whisper symbol")
    print("or flash_attn. Do not go on to Step 2. Show this to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: venv proof failed")
print("proof passed: transformers 4.44.2, loaded from the path above,")
print("WHISPER_ATTENTION_CLASSES imports from it, and flash_attn")
print("2.8.3 imports too.")

# The chip, read from the machine itself. This cell deliberately never
# imports torch or transformers into the notebook's own session.
smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print()
print((smi.stdout or smi.stderr).strip())
print("This notebook wants an L4 (about 24 GB). A T4 (about 16 GB)")
print("cannot hold the model in full form; the ladder would have to")
print("squeeze it, and the record would say so.")
print("Step 1 done. Go on to Step 2. No restart is needed.")

## Step 2 · Get the clips

All twelve clips are already in your Google Drive, in the folder named
`project-k-clips`: the five from the first Qwen runs, four swap cells,
and the three matched files. The next cell links your Drive, finds that
folder, and copies the clips to a plain work folder here.

The five older clips:

1. `P02_07_5.mp4`: the vacuum clip. A vacuum cleaner runs off camera.
2. `P32_01_54.mp4`: the jar clip. A jar washed under a running tap.
3. `own-01.mp4`: your eggs recording (cut to its first 15 seconds below
   if it is longer, exactly as the Qwen notebooks handled it).
4. `probe_speech.mp4`: the spoken sentence control.
5. `probe_tone.mp4`: three loud beeps, no voice.

The swap cells (one still picture, only the soundtrack changes):

6. `swap_speech.mp4`: the spoken sentence over the still picture.
7. `swap_beeps.mp4`: the beeps over the still picture.
8. `swap_water.mp4`: the jar's water sound over the still picture.
9. `swap_sizzle.mp4`: the eggs' sizzle over the still picture.

The matched files (same sentence, same speech loudness, beds added):

10. `matched_speech.mp4`: the sentence alone, the baseline.
11. `matched_speech_beeps.mp4`: the sentence over a beeps bed.
12. `matched_speech_vacuum.mp4`: the sentence over a vacuum bed.

The cell also checks each clip. Any clip longer than 20 seconds gets cut
to its first 15 seconds. At the end it prints which clips are ready and
which are missing. Do not go on until all twelve say "ready".

One thing this cell no longer does: check the notebook session's
transformers version. That check guarded the old design, where the
model ran inside this session. The model now runs under the private
environment's Python, whose versions Step 1 proved and the runner
re-proves on every run. This session's versions are simply not part of
the experiment anymore.

In [ ]:
# --- Get the clips -------------------------------------------------------------
# No version gate here anymore, on purpose. In the old design the model
# ran inside this session, so this session's transformers version could
# poison the run, and the gate caught exactly that on the second run.
# The model now runs under the private environment's Python (Step 1),
# which proves its own versions before anything loads. This cell only
# moves files.
import os, shutil, subprocess

WANT = ["swap_water.mp4", "swap_vacuum.mp4",
        "docs_applause_still.mp4", "swap_silence.mp4"]
WORK = "/content/clips"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = ["/content/drive/MyDrive/project-k-clips",   # Drive folder
           "/content/drive/MyDrive/docs-sounds-qwen3b",  # applause home
           "/content/project-k-clips"]                  # hand upload

for name in WANT:
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        continue
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            break

def duration_s(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path],
        capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

for name in WANT:
    path = os.path.join(WORK, name)
    if not os.path.exists(path):
        continue
    d = duration_s(path)
    if d is not None and d > 20:
        short = os.path.join(WORK, "short_" + name)
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", path, "-t", "15",
                        "-c:v", "libx264", "-pix_fmt", "yuv420p",
                        "-c:a", "aac", short], check=False)
        if os.path.exists(short):
            os.replace(short, path)
            print("cut", name, "to its first 15 seconds")

print()
ready = True
for name in WANT:
    path = os.path.join(WORK, name)
    if os.path.exists(path):
        print("ready:  ", name, f"({os.path.getsize(path) / 1e6:.1f} MB)")
    else:
        ready = False
        print("MISSING:", name)
if not ready:
    print()
    print("Some clips are missing. Check the Drive folder name is exactly")
    print("project-k-clips, or upload the files by hand, then run again.")

## Step 3 · Write the runner, then check: ladder and sanity

The model never runs inside this notebook's session. It runs in a
plain script, `/content/minicpm_run.py`, under the private
environment's Python, so every import it makes resolves inside the
environment Step 1 proved. The first cell below writes that script.
Nothing in it is new: it is the loading ladder, the sanity check, and
the battery from the earlier design, moved out of the session's reach.

The ladder, unchanged: three rungs, all of them the same model,
`openbmb/MiniCPM-o-2_6`, in three different fits. **Rung 1: full form**,
16-bit (bfloat16), loaded the model card's way, `AutoModel` with
`trust_remote_code=True`, sdpa attention, vision and audio parts on,
the speech-making part (TTS) left off to save about a gigabyte. On an
L4 this rung should load. **Rung 2: the 8-bit squeeze**, about 9 GB of
weights. **Rung 3: the 4-bit squeeze**, about 5 to 6 GB, the fit of
last resort. If all three fail, the script prints every reason and
stops. Whichever rung loads is printed and saved with every answer,
because a squeezed fit is a different run from the full form.

Three honesty bolts were added to the ladder in this round, after a
research pass proved the loading code itself correct against the real
checkpoint (all 1,599 weight tensors, zero shape mismatches) and the
only thing still weak was the evidence a failure leaves behind.
First, a failed rung now prints its **full traceback**, never a
cut-short reason, and the same words are appended to
`minicpm_load_log.txt` in the clips folder, so a verbatim record
survives the session; if all three rungs fail, the runner's closing
lines name that file. Second, the download is **pinned to one exact
hub revision**, the repo's head commit
`06849bfd36da94da1f5a88fa17e8ba63f08a4c4d`, so every run
loads byte-identical code and weights. Third, before loading, the
runner **clears this repo's cached remote code** (the copies
transformers keeps under the hub home), so code left by an older
session can never be reused against the pinned weights.

The second cell runs the script in **check mode**. It prints its proof
lines first (the versions and file it imported, and the
WHISPER_ATTENTION_CLASSES import), climbs the ladder, then asks the
sanity question: one plain text question, no video, answer printed in
full. That gate exists because of the first Qwen 3B run, where every
ask died on memory and the errors were stored as empty answers. If the
sanity answer comes back empty or with an error, the script stops and
Step 4 must not be run.

In [ ]:
# --- Write the runner script ---------------------------------------------------
# The runner holds the ladder, the sanity check, and the battery. It
# is written to a plain file here and only ever run under the private
# environment's Python, never imported into this session.
RUNNER_SOURCE = '\n# minicpm_run.py - the MiniCPM-o 2.6 runner for Project K.\n# Written by project_k_minicpm_o_colab.ipynb. Run only under the\n# private environment\'s Python (/content/minicpm-venv/bin/python).\n# Modes: "check" runs the proof, the loading ladder, and the sanity\n# question, then stops. "battery" does all of that and then asks the\n# twenty battery questions and saves the CSV.\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- Proof lines, printed before anything loads --------------------------------\n# The same proof Step 1 ran, repeated here so every run of this script\n# carries its own evidence: the versions it actually imported, from the\n# paths it actually loaded them, plus the one symbol the model\'s code\n# needs from transformers.\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nfrom transformers.models.whisper.modeling_whisper import WHISPER_ATTENTION_CLASSES\nprint("WHISPER_ATTENTION_CLASSES import: OK",\n      sorted(WHISPER_ATTENTION_CLASSES.keys()))\nif transformers.__version__ != "4.44.2":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.44.2.")\n    sys.exit(3)\n\n# --- Load the model: the MiniCPM-o ladder ---------------------------------------\n# Three rungs, all of them MiniCPM-o 2.6. A rung that fails prints its\n# full traceback (never a cut-short reason: an earlier design cut each\n# rung\'s error to its first 1200 characters, and the words that\n# mattered were exactly the ones cut) and the ladder climbs on. The\n# same traceback is appended, word for word, to the load log defined\n# below, so a failure leaves a verbatim record that survives this\n# session. If every rung fails, the run stops here and its closing\n# lines name the log file: no substitute model is ever loaded quietly.\nimport traceback, shutil, datetime\nfrom transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig\n\nREPO = "openbmb/MiniCPM-o-2_6"\n\n# The download is pinned to one exact hub revision: the repo\'s head\n# commit on Nov 1, 2026, when this round was built. Every run fetches\n# this commit, so the model\'s code and weights are byte-identical from\n# run to run, and an upstream change can never slip into a comparison\n# wearing the same model name.\nREVISION = "06849bfd36da94da1f5a88fa17e8ba63f08a4c4d"\n\n# The load log lives where the results CSV lives: the Drive clips\n# folder first, /content as the fallback, the same two homes in the\n# same order. Entries are appended as events happen, so the file is\n# already on disk even if the session dies right after a rung does.\nLOAD_LOG_PATHS = ["/content/drive/MyDrive/project-k-clips/minicpm_load_log.txt",\n                  "/content/minicpm_load_log.txt"]\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass   # that home is not mounted; the printed copy stands\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nlog_load("=" * 70 + "\\n")\nlog_load("MiniCPM-o 2.6 load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\nlog_load("repo: " + REPO + " at pinned revision " + REVISION + "\\n")\n\n# Clear this repo\'s cached remote code before loading. With\n# trust_remote_code, transformers copies the model\'s own Python files\n# into a cache under the hub home (the transformers dynamic-module\n# cache) and reuses them on later runs. A copy left there by an\n# earlier session could be stale next to the pinned weights, so this\n# repo\'s cached copy is deleted here and the pinned revision is\n# fetched fresh. Only this repo\'s folder is touched. The outcome,\n# whichever way it goes, is printed and written to the load log.\ntry:\n    from transformers.dynamic_module_utils import HF_MODULES_CACHE\n    module_cache = os.path.join(str(HF_MODULES_CACHE), *REPO.split("/"))\n    if os.path.isdir(module_cache):\n        shutil.rmtree(module_cache)\n        clear_note = "cleared cached remote code for this repo: " + module_cache\n    else:\n        clear_note = ("no cached remote code for this repo to clear: "\n                      + module_cache)\nexcept Exception as e:\n    clear_note = "WARNING: the remote-code cache clear did not run: " + str(e)\nprint(clear_note)\nlog_load(clear_note + "\\n")\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\ntokenizer = None\nMODEL_NAME = None   # repo name of the model that loaded\nLOAD_MODE = None    # the fit that worked: bf16, 8-bit, or 4-bit\n\ndef try_load(mode):\n    kwargs = dict(trust_remote_code=True, attn_implementation="sdpa",\n                  init_vision=True, init_audio=True, init_tts=False)\n    if mode == "bf16":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16, **kwargs)\n        m = m.eval().cuda()\n    elif mode == "8-bit":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16, load_in_8bit=True,\n            device_map="cuda", **kwargs)\n        m = m.eval()\n    elif mode == "4-bit":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16,\n            quantization_config=BitsAndBytesConfig(\n                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),\n            device_map="cuda", **kwargs)\n        m = m.eval()\n    t = AutoTokenizer.from_pretrained(REPO, revision=REVISION,\n                                      trust_remote_code=True)\n    return m, t\n\nrung_errors = []\nfor mode in ("bf16", "8-bit", "4-bit"):\n    print("=" * 70)\n    print(f"rung: {REPO} in {mode} form")\n    try:\n        m, t = try_load(mode)\n        model, tokenizer = m, t\n        MODEL_NAME, LOAD_MODE = REPO, mode\n        print("loaded:", REPO, "in", mode, "form")\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        rung_errors.append((REPO, mode, tb))\n        print("this rung failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("rung failed: " + REPO + " in " + mode + " form\\n")\n        log_load(tb)\n        model = None\n        tokenizer = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("ALL THREE RUNGS FAILED. MiniCPM-o 2.6 did not load in any fit.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every rung is saved, word for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nprint("environment: private venv /content/minicpm-venv, python 3.12,",\n      "torch 2.8.0+cu126, model card pin set")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\n# --- Sanity check: can the model write at all? ---------------------------------\n# This gate exists because of the first Qwen 3B run: every ask died on\n# memory, the errors were stored as empty answers, and the run looked\n# like a model that heard nothing. Nothing fails quietly here either.\n# Both modes run the sanity check; a failure stops the script before\n# any clip is asked.\ndef answer_text(res):\n    # model.chat hands back different shapes depending on mode and\n    # version: a plain string, a dict with a text field, or a tuple\n    # whose first item is the text. Normalise to the written words only.\n    if isinstance(res, dict):\n        return str(res.get("text", "") or "")\n    if isinstance(res, (list, tuple)):\n        return answer_text(res[0]) if res else ""\n    return str(res or "")\n\ntry:\n    msgs = [{"role": "user",\n             "content": ["Reply with the word ready."]}]\n    sres = model.chat(msgs=msgs, tokenizer=tokenizer,\n                      sampling=False, max_new_tokens=16)\n    sraw = answer_text(sres).strip()\n    print("SANITY RAW OUTPUT:", repr(sraw))\n    if sraw:\n        print("sanity check passed: the model can write.")\n    else:\n        print("WARNING: the sanity check came back EMPTY.")\n        print("STOP. Show this output to Muse.")\n        sys.exit(5)\nexcept SystemExit:\n    raise\nexcept Exception as e:\n    print("WARNING: the sanity check FAILED with an error:")\n    print(str(e))\n    print("STOP. Show this output to Muse.")\n    sys.exit(5)\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, ladder, and sanity all passed.")\n    print("The battery was not run. Run Step 4 for the questions.")\n    sys.exit(0)\n\n# --- Ask the questions: the battery ---------------------------------------------\nimport re, csv, json, math, glob, subprocess\nimport numpy as np\nimport librosa\nfrom PIL import Image\n\n# Kill-test prompts: byte-identical to the Pegasus kill test\n# (kill-test-wrapper/DESIGN-FROZEN.md). P2 and P3 are the battery forms.\nPROMPT_P2 = ("Describe what happens in this video, including any sounds "\n             "that are audible.")\nPROMPT_P3 = ("What sounds can you hear in this clip? Describe only what "\n             "is audible, without guessing from the pictures. If the "\n             "audio is silent, say \'silent\'.")\nPROMPT_R2 = "What happens in this video? Mention any sounds you can hear."\nPROMPT_R3 = "Summarize this video and list any audible sounds."\n\ndef yn_prompt(target):\n    return f"Is there the sound of {target} in this clip? Answer yes or no."\n\ndef w3_prompt(w1_answer):\n    return (f"A sound-only check of this same clip reported: \'{w1_answer}\' "\n            "Now describe what happens in this video, including any sounds "\n            "that are audible. Include the sounds from that sound-only "\n            "report in your answer.")\n\n# The feed, stated exactly. One frame per second, each frame downscaled to\n# fit inside 448x448, at most 15 one-second units, the matching second of\n# the 16 kHz sound track in every unit, one image slice per frame.\nFEED = ("omni chat units: 1 frame per second + the matching 1s of the "\n        "16 kHz audio track per unit, frames downscaled to fit 448x448, "\n        "max 15 units, max_slice_nums=1, greedy (sampling=False), "\n        "no generated speech")\nprint("FEED SETTINGS:", FEED)\nprint("MODEL:", MODEL_NAME, "in", LOAD_MODE, "form")\n\n# (sound, file, own target, lure target) - arms in the main loop below.\nKT_JOBS = [\n    ("water",    "swap_water.mp4",          "running water",    "a vacuum cleaner"),\n    ("vacuum",   "swap_vacuum.mp4",         "a vacuum cleaner", "applause"),\n    ("applause", "docs_applause_still.mp4", "applause",         "running water"),\n]\nKT_SILENCE = ("silence", "swap_silence.mp4", "running water", "")\n\nFRAME_DIR = "/content/frames"\nos.makedirs(FRAME_DIR, exist_ok=True)\n\ndef build_units(path):\n    # Build the model card\'s unit chain for one clip: for each second,\n    # "<unit>", one picture frame, that second of sound. ffmpeg does the\n    # cutting; the shape of what the model receives is the card\'s.\n    tag = re.sub(r"[^A-Za-z0-9]+", "_", os.path.basename(path))\n    fdir = os.path.join(FRAME_DIR, tag)\n    os.makedirs(fdir, exist_ok=True)\n    for old in glob.glob(os.path.join(fdir, "*.png")):\n        os.remove(old)\n    subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", path,\n                    "-vf", "fps=1", os.path.join(fdir, "f%03d.png")],\n                   check=False)\n    wav = os.path.join(fdir, "audio.wav")\n    subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", path,\n                    "-vn", "-ac", "1", "-ar", "16000", wav], check=False)\n    if os.path.exists(wav):\n        audio_np, sr = librosa.load(wav, sr=16000, mono=True)\n    else:\n        audio_np, sr = np.zeros(0, dtype=np.float32), 16000\n    frames = sorted(glob.glob(os.path.join(fdir, "*.png")))\n    num_units = int(math.ceil(len(audio_np) / sr)) if len(audio_np) else 0\n    num_units = max(num_units, len(frames))\n    num_units = min(num_units, 15)\n    contents = []\n    for i in range(num_units):\n        if frames:\n            img = Image.open(frames[min(i, len(frames) - 1)]).convert("RGB")\n            img.thumbnail((448, 448))\n        else:\n            img = Image.new("RGB", (448, 448))\n        chunk = audio_np[sr * i: sr * (i + 1)]\n        if len(chunk) < sr:\n            chunk = np.pad(chunk, (0, sr - len(chunk))).astype(np.float32)\n        contents.extend(["<unit>", img, chunk])\n    return contents\n\ndef ask(path, prompt):\n    sys_msg = model.get_sys_prompt(mode="omni", language="en")\n    contents = build_units(path)\n    contents = contents + [prompt]\n    msgs = [sys_msg, {"role": "user", "content": contents}]\n    res = model.chat(msgs=msgs, tokenizer=tokenizer,\n                     sampling=False, max_new_tokens=300,\n                     omni_input=True, use_tts_template=False,\n                     generate_audio=False,\n                     max_slice_nums=1, use_image_id=False)\n    text = answer_text(res).strip()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return text\n\ndef pull_fields(answer):\n    # Read seen / heard / sound_start_s out of a Part A answer. Whole JSON\n    # first, then a hunt for each field on its own. ok is False when\n    # nothing readable was found, so the caller shouts instead of saving\n    # silence.\n    m = re.search(r"\\{.*\\}", answer, re.S)\n    if m:\n        try:\n            obj = json.loads(m.group(0))\n            return (str(obj.get("seen", "")), str(obj.get("heard", "")),\n                    str(obj.get("sound_start_s", "")), True)\n        except Exception:\n            pass\n    def grab(key):\n        mm = re.search(r\'"%s"\\s*:\\s*"([^"]*)"\' % key, answer, re.S)\n        return mm.group(1) if mm else ""\n    seen, heard = grab("seen"), grab("heard")\n    ms = re.search(r\'"sound_start_s"\\s*:\\s*(null|[0-9.]+)\', answer)\n    start = ms.group(1) if ms else ""\n    if seen or heard or start:\n        return seen, heard, start, True\n    return "", "", "", False\n\nrows = []\nfailed = 0\n\ndef kt_ask(sound, name, arm, target, prompt):\n    global failed\n    path = os.path.join("/content/clips", name)\n    print("=" * 70)\n    print(f"{sound} - {arm} - {name}")\n    if not os.path.exists(path):\n        failed += 1\n        print("WARNING: clip missing, skipped. This run is incomplete.")\n        rows.append({"sound": sound, "file": name, "arm": arm,\n                     "target": target,\n                     "model": (f"{MODEL_NAME} ({LOAD_MODE}) "\n                               "[venv py3.12, torch 2.8.0+cu126]"),\n                     "prompt": prompt,\n                     "raw_text": "ERROR: clip missing", "feed": FEED})\n        return ""\n    try:\n        raw = ask(path, prompt)\n    except Exception as e:\n        raw = ""\n        error = "ERROR: " + str(e)\n        failed += 1\n        print(f"WARNING: {sound} {arm} FAILED with an error.")\n        print("The full error is saved in the CSV column raw_text.")\n        rows.append({"sound": sound, "file": name, "arm": arm,\n                     "target": target,\n                     "model": (f"{MODEL_NAME} ({LOAD_MODE}) "\n                               "[venv py3.12, torch 2.8.0+cu126]"),\n                     "prompt": prompt, "raw_text": error, "feed": FEED})\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n        return ""\n    if not raw.strip():\n        failed += 1\n        print(f"WARNING: {sound} {arm} came back EMPTY.")\n        rows.append({"sound": sound, "file": name, "arm": arm,\n                     "target": target,\n                     "model": (f"{MODEL_NAME} ({LOAD_MODE}) "\n                               "[venv py3.12, torch 2.8.0+cu126]"),\n                     "prompt": prompt,\n                     "raw_text": "ERROR: empty answer returned",\n                     "feed": FEED})\n        return ""\n    print(f"{arm} raw answer:")\n    print(raw)\n    rows.append({"sound": sound, "file": name, "arm": arm,\n                 "target": target,\n                 "model": (f"{MODEL_NAME} ({LOAD_MODE}) "\n                           "[venv py3.12, torch 2.8.0+cu126]"),\n                 "prompt": prompt, "raw_text": raw, "feed": FEED})\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return raw\n\n# Phase 1: every arm except W3 (W3 needs that clip\'s own W1 answer).\nw1_answers = {}\nfor sound, name, target, lure in KT_JOBS:\n    kt_ask(sound, name, "S", target, PROMPT_P2)\n    kt_ask(sound, name, "R1", target, PROMPT_P2)\n    kt_ask(sound, name, "R2", target, PROMPT_R2)\n    kt_ask(sound, name, "R3", target, PROMPT_R3)\n    w1_answers[sound] = kt_ask(sound, name, "W1", target, PROMPT_P3)\n    kt_ask(sound, name, "W2", target, yn_prompt(target))\n    kt_ask(sound, name, "LURE", lure, yn_prompt(lure))\ns_sound, s_name, s_target, _ = KT_SILENCE\nkt_ask(s_sound, s_name, "S", s_target, PROMPT_P2)\nw1_answers[s_sound] = kt_ask(s_sound, s_name, "W1", s_target, PROMPT_P3)\nkt_ask(s_sound, s_name, "W2", s_target, yn_prompt(s_target))\n# Phase 2: W3 carry-forward finals.\nfor sound, name, target, lure in KT_JOBS + [KT_SILENCE]:\n    kt_ask(sound, name, "W3", target, w3_prompt(w1_answers.get(sound, "")))\n\nif failed:\n    print()\n    print("!" * 70)\n    print(f"RUN PROBLEM: {failed} ask(s) failed, came back empty, or a")\n    print("clip was missing. Their raw words or errors are in the CSV")\n    print("column raw_text. This run is NOT clean evidence until those")\n    print("asks are explained. Show this to Muse.")\n    print("!" * 70)\nelse:\n    print()\n    print("Clean run: every ask answered.")\n\nFIELDS = ["sound", "file", "arm", "target", "model", "prompt",\n          "raw_text", "feed"]\nfor out_path in ("/content/drive/MyDrive/project-k-clips/killtest_minicpm_results.csv",\n                 "/content/killtest_minicpm_results.csv"):\n    try:\n        with open(out_path, "w", newline="") as fh:\n            w = csv.DictWriter(fh, fieldnames=FIELDS)\n            w.writeheader()\n            w.writerows(rows)\n        print("saved:", out_path)\n    except Exception as e:\n        print("could not save", out_path, "-", str(e)[:120])\n\nif failed:\n    sys.exit(6)\n'
with open("/content/minicpm_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/minicpm_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, ladder, sanity -------------------------------------------
# Runs the runner under the private environment's Python and streams
# its output here live. A nonzero exit code means the proof, the
# ladder, or the sanity gate failed, and the run stops in this cell.
import os, subprocess
VENV_PY = "/content/minicpm-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/minicpm_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, a ladder rung, or the sanity question.")
    print("Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, sanity answered.")
print("Go on to Step 4.")

## Step 4 · Ask the questions: the battery

The next cell runs the same runner script in **battery mode**: proof
lines again, ladder again, sanity again, then the twenty questions.
Two parts, the same battery the other models faced.

**Part A: the repaired JSON question**, word for word the one Pegasus and
both Qwens got. Seven clips, two repeats each: the vacuum, the jar, the
eggs, and four swap cells (speech, beeps, water, sizzle, each a
soundtrack over one still picture that never changes). The model must
answer in the fixed shape: what it can SEE, what it can HEAR, and the
second the main sound starts, with "none" and null allowed as honest
answers.

**Part B: the sounds-only question**, word for word the P3 question from
the Pegasus matched run. Three matched files, two repeats each: the
spoken sentence alone, the same sentence over a beeps bed, and the same
sentence over a vacuum bed, the speech at the same loudness in all three.
These answers are free text, saved in full. The sentence to check them
against is: "The blue kettle boils at seven forty two."

**How this model is asked.** MiniCPM-o does not take a video file the way
the Qwen path did. It takes the clip as a chain of one-second units, the
pattern its own model card documents: for each second, one picture frame
plus that second of the sound track, then the question at the end. The
runner builds those units with ffmpeg: frames at one per second,
downscaled to fit inside 448 by 448, at most 15 units per clip, and the
sound track pulled out whole at 16 kHz and sliced into the same seconds.
The model is asked through `model.chat` in omni mode with the omni
system prompt, greedy decoding (no sampling), and it is never asked to
generate speech. The exact feed settings are printed by the runner, saved
in every row of the results file, and belong in any comparison: this
model was fed differently from the Qwens, and that difference is part of
the record.

Nothing fails quietly. The full raw words of every ask are saved in the
`raw_text` column. An ask that errors is saved with its full error text
in that column and a loud WARNING on screen, never as an empty answer.
If you see a WARNING or the RUN PROBLEM banner, this run is not clean
evidence. Show it to Muse.

At the end, every answer is saved to `killtest_minicpm_results.csv` in the same
Drive folder the clips came from (the Drive mount from Step 2 is a
filesystem mount, so the runner can write there directly), and the three
Part B answers print in full so the sentence comparison can be read right
off the screen.

In [ ]:
# --- Battery mode: the twenty questions ------------------------------------------
# Same runner, same private Python, streamed live. The runner saves the
# CSV itself, to the Drive clips folder and to /content. A nonzero exit
# code means the pre-flight gates failed up front, or the run finished
# with broken asks (the RUN PROBLEM banner above says which).
import os, subprocess
VENV_PY = "/content/minicpm-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/minicpm_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: either a gate failed before the questions, or")
    print("some asks are broken and marked in the CSV. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: every ask answered and the CSV is saved.")

## Closing · How to read the answers

Read each cell against the other three models. The questions were
identical and the files were identical, so a difference is a model or
feed difference, and the feed difference is written down, not hidden.

- **Warnings first.** If the sanity check failed, or the battery printed
  any WARNING, this run has broken rows, marked in the CSV with their
  raw words or errors in `raw_text`. Broken rows are not evidence.
- **Part A, real clips.** Pegasus reported no sound on the vacuum, the
  jar, and the eggs. The Qwen 3B heard running water on the jar and
  nothing on the eggs; the 7B heard the eggs and the jar, and guessed
  kitchen noise on the vacuum clip. Where does a fourth family land?
- **Part A, swap cells.** One picture, four soundtracks. Pegasus
  reported only the speech. The 7B heard the beeps and the water, missed
  the sizzle, and, under the box question, missed the speech too.
- **Part B, matched files.** Pegasus returned the sentence perfectly
  alone and over beeps, and bent it over the vacuum bed ("Canon boils at
  seven forty two") while still saying no other sounds were audible. The
  7B failed the baseline itself (it called plain speech "silent"), so
  its bed rows are void and prove nothing. MiniCPM's Part B stands or
  falls on its own baseline the same way: read the alone file first.
  Only if the sentence comes back whole there do the two bed files mean
  anything.
- **The input honesty line.** This model was fed one frame per second
  with the matching second of sound, frames inside 448 by 448. The
  Qwens were fed one frame per second with frames capped smaller, and
  the full sound track. Pegasus received the whole file through the
  Twelve Labs API. So a MiniCPM miss could still be a feed difference;
  a MiniCPM hit on sound cannot be explained by a richer picture,
  because its picture feed was the plainest of the three.
- **The environment honesty line.** This model ran in a private virtual
  environment of its own (Python 3.12, torch 2.8.0+cu126, the model
  card's pinned transformers 4.44.2), because Colab's shared
  environment defeated two attempts to install that stack into it. The
  environment string is saved in the `model` column of every CSV row.
  The Qwen runs used Colab's own stack of their day. An environment is
  part of a run's record, the same way a feed is; it is stated here so
  nobody has to dig for it.

When you finish, the file `killtest_minicpm_results.csv` in your Drive is the
record. Send it back, and the four runs get written up side by side.